# The bystander interferon response



In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
TABLES = Path("../results/tables")
FIGURES = Path("../results/figures")
TABLES.mkdir(parents=True, exist_ok=True)

from perturbseq.data import load_cells
from perturbseq.estimators import bh_fdr
from perturbseq.genesets import IFN_PRODUCING
from perturbseq.genesets import ISG_GENES
import scanpy as sc

*from `06_isg.py`*

06 - Reproduction of the interferon panel (paper Fig. 5): bystander ISG scores.

Writes
    results/tables/06_isg_by_element.tsv
    results/tables/06_isg_hits.tsv
    results/tables/06_isg_paper_comparison.tsv

The published analysis restricts the ISG score to bystander cells, because
infected cells actively suppress interferon signalling, so a score computed on
a mixed population mostly measures each element's infected fraction rather than
its effect on interferon. That restriction is kept here, and the script shows what happens
without it.

Two traps, both handled explicitly:

1. Score self-containment. Several ISGs are themselves CRISPRi targets in this
   library (STAT1, STAT2, IRF9, BST2, among others). Knocking a gene down
   mechanically lowers any score that contains it, which looks like a signalling
   phenotype but is just the on-target effect. The score is therefore computed
   both with and without the element's own target genes, and the table carries
   both.

2. Depth. The score is a difference of means of log1p host CPM against an
   expression-matched background, so it is depth-normalised by construction,
   but bystander depth still varies by element. The per-element test keeps lane
   as a stratum and the null is lane- and size-matched, as in 07.

In [2]:
B_DRAWS = 2000

MIN_BYSTANDER = 30

RNG = np.random.default_rng(0)

PAPER_ISG_DOWN = ["GPR89A", "GPR89B", "GPR89A+GPR89B", "KEAP1", "SPNS1", "BRD2"]

PAPER_IFN_PATHWAY = ["IFNAR1", "IFNAR2", "STAT1", "STAT2", "IRF3", "IRF9"]

a = load_cells(assigned_only=True)
state = pd.read_csv(TABLES / "03_cell_state.tsv.gz", sep="\t").set_index("cell_barcode")
a.obs["state"] = state.reindex(a.obs_names)["state"].to_numpy()

In [3]:
# ------------------------------------------------- host-only normalisation
host = (a.var.kind == "host").to_numpy()
b = a[:, host].copy()
sc.pp.normalize_total(b, target_sum=1e6)   # host-only denominator
sc.pp.log1p(b)

present = [g for g in ISG_GENES if g in b.var_names]
print(f"ISG set: {len(present)}/{len(ISG_GENES)} genes present")
sc.tl.score_genes(b, present, score_name="isg", random_state=0)
a.obs["isg"] = b.obs.isg.to_numpy()

# ISG score excluding any gene that is a CRISPRi target in this library
targets_in_library = set()
for t in a.obs.target.unique():
    if t not in ("NTC", "unassigned"):
        targets_in_library.update(t.split("+"))
clean = [g for g in present if g not in targets_in_library]
dropped = sorted(set(present) - set(clean))
print(f"ISGs that are also CRISPRi targets, dropped from the clean score: {dropped}")
sc.tl.score_genes(b, clean, score_name="isg_clean", random_state=0)
a.obs["isg_clean"] = b.obs.isg_clean.to_numpy()

# the rare interferon-producing subset the paper describes
ifn_present = [g for g in IFN_PRODUCING if g in b.var_names]
a.obs["ifn_producing_umi"] = np.asarray(
    a[:, ifn_present].X.sum(axis=1)
).ravel() if ifn_present else 0.0

d = a.obs.copy()
d["gem_group"] = d.gem_group.astype(str)
byst = d[d.state == "uninfected"]
inf = d[d.state == "infected"]
print(
    f"\nISG score: bystander mean {byst.isg.mean():.4f}, infected mean {inf.isg.mean():.4f} "
    f"(the paper reports infected cells lower, i.e. active suppression)"
)
print(
    f"interferon-producing cells (>=5 UMI of {len(ifn_present)} IFN/chemokine genes): "
    f"bystander {(byst.ifn_producing_umi >= 5).mean() * 100:.2f}%, "
    f"infected {(inf.ifn_producing_umi >= 5).mean() * 100:.2f}%"
)

ISG set: 55/55 genes present


ISGs that are also CRISPRi targets, dropped from the clean score: ['BST2', 'DDX58', 'EIF2AK2', 'IFIH1', 'IFITM1', 'IFITM2', 'IFITM3', 'IRF9', 'ISG15', 'LY6E', 'STAT1', 'STAT2']



ISG score: bystander mean 2.4066, infected mean 1.0402 (the paper reports infected cells lower, i.e. active suppression)
interferon-producing cells (>=5 UMI of 10 IFN/chemokine genes): bystander 39.43%, infected 19.91%


In [4]:
# ------------------------------------------- per-element test in bystanders
lanes = sorted(byst.gem_group.unique())
rows = []
for score_col in ("isg", "isg_clean"):
    ntc = byst[byst.is_ntc]
    pool = {l: ntc.loc[ntc.gem_group == l, score_col].to_numpy() for l in lanes}
    base = {l: float(np.mean(v)) if len(v) else np.nan for l, v in pool.items()}

    for element, sub in byst.groupby("element", observed=True):
        if len(sub) < MIN_BYSTANDER:
            continue
        n_l, num, w = [], 0.0, 0.0
        for l in lanes:
            s = sub.loc[sub.gem_group == l, score_col].to_numpy()
            if len(s) == 0 or not np.isfinite(base[l]):
                n_l.append(0)
                continue
            n_l.append(len(s))
            num += len(s) * (s.mean() - base[l])
            w += len(s)
        if w == 0:
            continue
        obs = num / w
        null = np.empty(B_DRAWS)
        for i in range(B_DRAWS):
            nn = 0.0
            for l, k in zip(lanes, n_l):
                if k == 0:
                    continue
                p = pool[l]
                nn += k * (p[RNG.integers(0, len(p), k)].mean() - base[l])
            null[i] = nn / w
        sd = float(np.std(null))
        rows.append(
            dict(
                score=score_col,
                element=element,
                target=sub.target.iloc[0],
                is_ntc=bool(sub.is_ntc.iloc[0]),
                n_bystander=int(len(sub)),
                delta_isg=obs,
                null_sd=sd,
                z_empirical=obs / sd if sd > 0 else np.nan,
                p_null=(1 + int(np.sum(np.abs(null) >= abs(obs)))) / (B_DRAWS + 1),
            )
        )
out = pd.DataFrame(rows)
out["q_null"] = np.nan
for sc_name, idx in out.groupby("score").groups.items():
    out.loc[idx, "q_null"] = bh_fdr(out.loc[idx, "p_null"].to_numpy())
out = out.sort_values(["score", "delta_isg"])
out.to_csv(TABLES / "06_isg_by_element.tsv", sep="\t", index=False)

main_tab = out[out.score == "isg_clean"]
tested = main_tab[~main_tab.is_ntc]
sig = tested[tested.q_null < 0.1]
print(
    f"\nbystander ISG, target-free score: {len(tested)} elements tested, "
    f"{len(sig)} at q<0.1 ({int((sig.delta_isg < 0).sum())} down, "
    f"{int((sig.delta_isg > 0).sum())} up)"
)
print("\nlargest decreases:")
print(
    tested.nsmallest(15, "delta_isg")[
        ["element", "target", "n_bystander", "delta_isg", "z_empirical", "q_null"]
    ].round(4).to_string(index=False)
)
print("\nlargest increases:")
print(
    tested.nlargest(8, "delta_isg")[
        ["element", "target", "n_bystander", "delta_isg", "z_empirical", "q_null"]
    ].round(4).to_string(index=False)
)
sig.to_csv(TABLES / "06_isg_hits.tsv", sep="\t", index=False)


bystander ISG, target-free score: 170 elements tested, 23 at q<0.1 (20 down, 3 up)

largest decreases:
                  element        target  n_bystander  delta_isg  z_empirical  q_null
        CRISPRi_121_STAT2         STAT2           67    -3.2539     -31.6169  0.0059
  CRISPRi_181_STAT1_STAT2   STAT1+STAT2           82    -3.2323     -35.5919  0.0059
         CRISPRi_071_IRF9          IRF9           74    -3.2241     -33.5599  0.0059
    CRISPRi_182_IRF9_IRF3     IRF9+IRF3           75    -2.6218     -27.2785  0.0059
CRISPRi_180_IFNAR1_IFNAR2 IFNAR1+IFNAR2           89    -2.2870     -25.5682  0.0059
         CRISPRi_138_TYK2          TYK2           60    -2.1781     -20.3816  0.0059
       CRISPRi_069_IFNAR2        IFNAR2          101    -2.0800     -24.8691  0.0059
       CRISPRi_162_GPR89A        GPR89A           58    -1.9016     -17.4995  0.0059
       CRISPRi_068_IFNAR1        IFNAR1           60    -1.8545     -18.1289  0.0059
CRISPRi_193_GPR89A_GPR89B GPR89A+GPR89B       

In [5]:
# ------------------------------------------------- against the paper's list
rows = []
for tgt in PAPER_ISG_DOWN + PAPER_IFN_PATHWAY:
    for score_col in ("isg", "isg_clean"):
        sub = out[(out.target == tgt) & (out.score == score_col)]
        if sub.empty:
            rows.append(dict(target=tgt, score=score_col, note="not testable"))
            continue
        for _, r in sub.iterrows():
            n = out[(out.score == score_col) & (~out.is_ntc)]
            rows.append(
                dict(
                    target=tgt,
                    score=score_col,
                    element=r.element,
                    paper=("down" if tgt in PAPER_ISG_DOWN else "pathway group"),
                    n_bystander=int(r.n_bystander),
                    delta_isg=r.delta_isg,
                    q_null=r.q_null,
                    rank=int((n.delta_isg < r.delta_isg).sum() + 1),
                    n_tested=len(n),
                )
            )
cmp = pd.DataFrame(rows)
cmp.to_csv(TABLES / "06_isg_paper_comparison.tsv", sep="\t", index=False)
print("\nthe paper's interferon results in this ranking:")
print(cmp.round(4).to_string(index=False))


the paper's interferon results in this ranking:
       target     score                   element         paper  n_bystander  delta_isg  q_null  rank  n_tested
       GPR89A       isg        CRISPRi_162_GPR89A          down           58    -1.8485  0.0059     9       170
       GPR89A isg_clean        CRISPRi_162_GPR89A          down           58    -1.9016  0.0059     8       170
       GPR89B       isg        CRISPRi_163_GPR89B          down           84     0.0272  0.9244   118       170
       GPR89B isg_clean        CRISPRi_163_GPR89B          down           84     0.0524  0.8156   124       170
GPR89A+GPR89B       isg CRISPRi_193_GPR89A_GPR89B          down           93    -1.5979  0.0059    10       170
GPR89A+GPR89B isg_clean CRISPRi_193_GPR89A_GPR89B          down           93    -1.6249  0.0059    10       170
        KEAP1       isg         CRISPRi_164_KEAP1          down           72    -0.7470  0.0059    13       170
        KEAP1 isg_clean         CRISPRi_164_KEAP1      